In [1]:
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import muon as mu

import time
import sys
import scipy
import os
import scipy
import phate

from pathlib import Path

import Utils as ut

/home/gabriele/miniconda3/envs/tda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/gabriele/miniconda3/envs/tda/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
adata=sc.read_h5ad("../Topology/Datasets/PBMC/CM/PBMC_GEX_Def.h5ad")
adata

AnnData object with n_obs × n_vars = 6169 × 27607
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'mt_frac', 'ribofrac', 'passes_filter', 'doublet_score', 'predicted_doublet'
    var: 'gene_ids', 'feature_types', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'scrublet'
    layers: 'counts'

In [3]:
adata_red=adata[:1000, :500].copy()
adata_red

AnnData object with n_obs × n_vars = 1000 × 500
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'mt_frac', 'ribofrac', 'passes_filter', 'doublet_score', 'predicted_doublet'
    var: 'gene_ids', 'feature_types', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg', 'scrublet'
    layers: 'counts'

In [4]:
from scTDA import scTDA

2026-06-01 15:58:04,019 - INFO - ❌ CUDA not available. Using CPU.


Module base_dataset has been imported DEVICE --> cpu


In [5]:
out1=scTDA(adata_red, embedder="GRAE")

2026-06-01 15:58:04,033 - INFO - Fitting manifold learning method...


Calculating PHATE...
  Running PHATE on 850 observations and 500 variables.
  Calculating graph and diffusion operator...
    Calculating PCA...
    Calculated PCA in 0.35 seconds.
    Calculating KNN search...
    Calculated KNN search in 0.10 seconds.
    Calculating affinities...
    Calculated affinities in 0.52 seconds.
  Calculated graph and diffusion operator in 0.98 seconds.
  Calculating optimal t...
    Automatically selected t = 9
  Calculated optimal t in 0.15 seconds.
  Calculating diffusion potential...
  Calculated diffusion potential in 0.11 seconds.
  Calculating metric MDS...
  Calculated metric MDS in 3.41 seconds.
Calculated PHATE in 4.65 seconds.


2026-06-01 15:58:08,690 - INFO - Fitting encoder & decoder...
2026-06-01 15:58:10,084 - INFO - Epoch 10...
2026-06-01 15:58:10,901 - INFO - Epoch 20...
2026-06-01 15:58:11,980 - INFO - Epoch 30...
2026-06-01 15:58:12,315 - INFO - Model has been early stopped at epoch 34


In [4]:
marker_genes_by_celltype = {
    'T': [
        'IL7R', 'TRAC', 'GATA3',
        'LEF1', 'FHIT', 'RORA', 'ITGB1',
        'CD8A', 'CD8B', 'CD248', 'CCL5',
        'GZMH', 'GZMK',
        'KLRB1', 'SLC4A10',
        'IL32'
    ],

    'NK': [
        'GNLY', 'NKG7',
        'IL32'
    ],

    'B': [
        'CD79A', 'MS4A1', 'IGHD', 'IGHM',
        'IL4R', 'TNFRSF13C'
    ],

    'Monocytes': [
        'KLF4', 'LYZ', 'S100A8', 'ITGAM', 'CD14',
        'DPYD',
        'FCGR3A', 'MS4A7', 'CST3'
    ],

    'DC': [
        'CLEC10A', 'IRF8', 'TCF4'
    ]
}

In [15]:
adata=sc.read_h5ad("Datasets/PBMC/CM/PBMC_GEX_Def.h5ad")
#ad_red=adata[adata.obs.Raw_CellType=="B", adata.var.highly_variable==True][:, 1200:1800].copy()

In [16]:
ads={}
mrks={}
for key in marker_genes_by_celltype.keys():
    ad_red=adata[adata.obs.Raw_CellType==key, adata.var.highly_variable==True].copy()
    ads[key]=ad_red
    ad_red.uns["Markers_for_TDA"]=marker_genes_by_celltype[key]
    present = [g for g in marker_genes_by_celltype[key] if g in ad_red.var_names]
    
    # indices in current order
    idx = [ad_red.var_names.get_loc(g) for g in present]
    print(dict(zip(present, idx)))
    
    # reorder so these genes are first
    new_order = present + [g for g in ad_red.var_names if g not in present]
    
    ad_red = ad_red[:, new_order]

    os.system(f"mkdir -p Datasets/PBMC/TDA_CellType_GEX/{key}")
    ad_red.write(f"Datasets/PBMC/TDA_CellType_GEX/{key}/PBMC_GEX_{key}.h5ad", compression="gzip")

T (3161, 2000) [261, 262, 296, 1674, 1139, 1308, 636, 1469]
{'GATA3': 1139, 'CD8A': 261, 'CD8B': 262, 'CCL5': 1674, 'GZMH': 1469, 'GZMK': 636, 'KLRB1': 1308, 'SLC4A10': 296}
NK (267, 2000) [258, 1810]
{'GNLY': 258, 'NKG7': 1810}
B (626, 2000) [1245, 1799]
{'CD79A': 1799, 'MS4A1': 1245}
Monocytes (2014, 2000) [106, 1244]
{'FCGR3A': 106, 'MS4A7': 1244}
DC (101, 2000) [1746, 1654]
{'CLEC10A': 1654, 'TCF4': 1746}


In [2]:
import os
for key in ["T"]:
    os.system(f"sbatch -p slim18 -J TDA{key} --mem=200G --ntasks=1 --cpus-per-task=8 --time 7-00:00:00 --exclusive\
                --output=Datasets/PBMC/PBMC_{key}_Run_scTDA.out Run_Exp.sh Run_scTDA Datasets/PBMC/TDA_CellType/{key}/PBMC_GEX_{key}.h5ad Datasets/PBMC/TDA_CellType/{key} {key}")

Submitted batch job 1551559


In [9]:
import os

for key in ["T","B","Monocytes"]:
    os.system(f"sbatch -p slim18 -J xgTDA{key} --mem=48G --ntasks=1 --cpus-per-task=8 --time 7-00:00:00 --exclusive\
               --output=Datasets/PBMC/PBMC_{key}_Run_Global_scTDA.out Run_Exp.sh Run_scTDA_global \
               Datasets/PBMC/TDA_CellType_GEX/{key}/PBMC_GEX_{key}.h5ad Datasets/PBMC/TDA_CellType/{key}/Global {key}")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Submitted batch job 1580025
Submitted batch job 1580026
Submitted batch job 1580027


In [10]:
os.system(f"cat Datasets/PBMC/PBMC_{key}_Run_Global_scTDA.out")

0

In [9]:
adata=sc.read_h5ad("Datasets/PBMC/CM/PBMC_Peak_Def.h5ad")
adata
#ad_red=adata[adata.obs.Raw_CellType=="B", adata.var.highly_variable==True][:, 1200:1800].copy()

AnnData object with n_obs × n_vars = 6180 × 63906
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType', 'nucleosome_signal', 'tss_enrichment_score', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'passes_filter'
    var: 'chr', 'start', 'stop', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'fragment_lengths', 'hvg', 'tss_enrichment'
    layers: 'counts'

In [16]:
ads={}
mrks={}
for key in marker_genes_by_celltype.keys():
    print(key)
    ad_red=adata[adata.obs.Raw_CellType==key, adata.var.highly_variable==True].copy()
    ads[key]=ad_red
    ad_red.uns["Markers_for_TDA"]=marker_genes_by_celltype[key] 
    # reorder so these genes are first    
    os.system(f"mkdir -p Datasets/PBMC/TDA_CellType_Peak/{key}")
    ad_red.write(f"Datasets/PBMC/TDA_CellType_Peak/{key}/PBMC_Peak_{key}.h5ad", compression="gzip")

T
NK
B
Monocytes
DC


In [24]:
import os
for key in marker_genes_by_celltype.keys():
    print(key)
    os.system(f"sbatch -p slim18 -J TDA{key} --mem=64G --ntasks=1 --cpus-per-task=8 --time 7-00:00:00 --exclusive\
                --output=Datasets/PBMC/PBMC_{key}_Run_scTDA_Peak.out Run_Exp.sh Run_scTDA_Peak Datasets/PBMC/TDA_CellType_Peak/{key}/PBMC_Peak_{key}.h5ad Datasets/PBMC/TDA_CellType_Peak/{key} {key}")

T
Submitted batch job 1554113
NK
Submitted batch job 1554114
B
Submitted batch job 1554115
Monocytes
Submitted batch job 1554116
DC
Submitted batch job 1554117


Raw_CellType
T            3300
Monocytes    1896
B             590
NK            258
DC            136
Name: count, dtype: int64

# HumanBrain

## GEX

In [3]:
adata=sc.read_h5ad("../GNN/Datasets/10XhsBrain3kMO/FeatureSpaces/GEX/CM/10XhsBrain3kMO_GEX_Raw.h5ad")
adata

AnnData object with n_obs × n_vars = 3175 × 43925
    obs: 'CellType', 'Raw_celltype'
    var: 'gene_ids', 'feature_types'

In [4]:
adata=ut.qc_filtering(adata, omic="GEX")

Adata's shape: (3175, 43925)
Adata's shape after cells and features filtering: (2857, 30027)
Adata's shape after MT filtering: (2857, 30027)
Adata's shape after doublets filtering: (2848, 30027)
Adata's shape after HVG filtering: (2848, 30027)


In [5]:
adata

AnnData object with n_obs × n_vars = 2848 × 30027
    obs: 'CellType', 'Raw_celltype', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'mt_frac', 'ribofrac', 'passes_filter', 'doublet_score', 'predicted_doublet'
    var: 'gene_ids', 'feature_types', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'scrublet', 'hvg'
    layers: 'counts'

In [7]:
adata.write("Datasets/HumanBrain/CM/HumanBrain_GEX_Def.h5ad", compression="gzip")

## Peak

In [9]:
adata=sc.read_h5ad("../GNN/Datasets/10XhsBrain3kMO/FeatureSpaces/Peak/CM/10XhsBrain3kMO_Peak_Raw.h5ad")
adata

AnnData object with n_obs × n_vars = 3175 × 174200
    obs: 'CellType', 'Raw_celltype', 'nucleosome_signal', 'tss_enrichment_score'
    var: 'chr', 'start', 'stop'
    uns: 'fragment_lengths', 'tss_enrichment'

In [10]:
adata=ut.qc_filtering(adata, omic="ATAC")

Adata's shape: (3175, 174200)
Adata's shape after cells and features filtering: (2857, 89430)
Adata's shape after HVG filtering: (2857, 89430)


In [11]:
adata

AnnData object with n_obs × n_vars = 2857 × 89430
    obs: 'CellType', 'Raw_celltype', 'nucleosome_signal', 'tss_enrichment_score', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'passes_filter'
    var: 'chr', 'start', 'stop', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'fragment_lengths', 'tss_enrichment', 'hvg'
    layers: 'counts'

In [12]:
adata.write("Datasets/HumanBrain/CM/HumanBrain_Peak_Def.h5ad", compression="gzip")

## Muon obj

In [ ]:
import muon as mu
peak=sc.read_h5ad(f"Datasets/HumanBrain/CM/HumanBrain_GEX_Def.h5ad")
gex=sc.read_h5ad(f"Datasets/HumanBrain/CM/HumanBrain_Peak_Def.h5ad")
#Peak + GEX
print("Peak + GEX")
inter=ut.intersection([peak.obs.index, gex.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "GEX" : gex[inter]})
mudata.write_h5mu(f"Datasets/HumanBrain/CM/HumanBrain_Peak_GEX_Def.h5mu", compression="gzip")

Peak + GEX


In [16]:
mudata

MuData object with n_obs × n_vars = 2629 × 119457
  var:	'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
  2 modalities
    Peak:	2629 x 30027
      obs:	'CellType', 'Raw_celltype', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'mt_frac', 'ribofrac', 'passes_filter', 'doublet_score', 'predicted_doublet'
      var:	'gene_ids', 'feature_types', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'hvg', 'scrublet'
      layers:	'counts'
    GEX:	2629 x 89430
      obs:	'CellType', 'Raw_celltype', 'nucleosome_signal', 'tss_enrichment_score', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'passes_filter'
      var:	'chr', 'start', 'stop', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'fragment_lengths', 'hvg', 'tss_enrichment'
      layers:	'counts'

# PBMC

## GEX

In [20]:
adata=sc.read_h5ad("../SEAGALL/Datasets/10XhsPBMC10kMO/FeatureSpaces/GEX/CM/10XhsPBMC10kMO_GEX_Raw.h5ad")
adata

AnnData object with n_obs × n_vars = 7082 × 43925
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType'
    var: 'gene_ids', 'feature_types'

In [ ]:
adata=ut.qc_filtering(adata, omic="GEX")

In [19]:
adata

AnnData object with n_obs × n_vars = 6365 × 27607
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'mt_frac', 'ribofrac', 'passes_filter', 'doublet_score', 'predicted_doublet'
    var: 'gene_ids', 'feature_types', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'scrublet', 'hvg'
    layers: 'counts'

In [22]:
# create coarse cell-type annotation
adata.obs["Raw_CellType"] = pd.Series(index=adata.obs.index, dtype="object")

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^B_", regex=True),
    "Raw_CellType"
] = "B"

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^T_", regex=True),
    "Raw_CellType"
] = "T"

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^NK$", regex=True),
    "Raw_CellType"
] = "NK"

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^Monocytes$", regex=True),
    "Raw_CellType"
] = "Monocytes"

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^DC", regex=True),
    "Raw_CellType"
] = "DC"

# optional: remove cells not assigned
adata = adata[adata.obs["Raw_CellType"].notna()].copy()

# check result
adata.obs["Raw_CellType"].value_counts()

Raw_CellType
T            3161
Monocytes    2014
B             626
NK            267
DC            101
Name: count, dtype: int64

In [23]:
adata.write("Datasets/PBMC/CM/PBMC_GEX_Def.h5ad", compression="gzip")

## Peak

In [3]:
adata=sc.read_h5ad("../SEAGALL/Datasets/10XhsPBMC10kMO/FeatureSpaces/Peak/CM/10XhsPBMC10kMO_Peak_Raw.h5ad")
adata

AnnData object with n_obs × n_vars = 7082 × 129625
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType', 'nucleosome_signal', 'tss_enrichment_score'
    var: 'chr', 'start', 'stop'
    uns: 'fragment_lengths', 'tss_enrichment'

In [4]:
adata=ut.qc_filtering(adata, omic="ATAC")

Adata's shape: (7082, 129625)
Adata's shape after cells and features filtering: (6372, 63906)
Adata's shape after HVG filtering: (6372, 63906)


In [5]:
adata

AnnData object with n_obs × n_vars = 6372 × 63906
    obs: 'RNA_celltype', 'Raw_CellType', 'CellType', 'nucleosome_signal', 'tss_enrichment_score', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'passes_filter'
    var: 'chr', 'start', 'stop', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'fragment_lengths', 'tss_enrichment', 'hvg'
    layers: 'counts'

In [6]:
# create coarse cell-type annotation
adata.obs["Raw_CellType"] = pd.Series(index=adata.obs.index, dtype="object")

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^B_", regex=True),
    "Raw_CellType"
] = "B"

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^T_", regex=True),
    "Raw_CellType"
] = "T"

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^NK$", regex=True),
    "Raw_CellType"
] = "NK"

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^Monocytes$", regex=True),
    "Raw_CellType"
] = "Monocytes"

adata.obs.loc[
    adata.obs["CellType"].str.contains(r"^DC", regex=True),
    "Raw_CellType"
] = "DC"

# optional: remove cells not assigned
adata = adata[adata.obs["Raw_CellType"].notna()].copy()

# check result
adata.obs["Raw_CellType"].value_counts()

Raw_CellType
T            3300
Monocytes    1896
B             590
NK            258
DC            136
Name: count, dtype: int64

In [8]:
adata.write("Datasets/PBMC/CM/PBMC_Peak_Def.h5ad", compression="gzip")

## Muon object

In [14]:
gex=sc.read_h5ad(f"Datasets/PBMC/CM/PBMC_GEX_Def.h5ad")
peak=sc.read_h5ad(f"Datasets/PBMC/CM/PBMC_Peak_Def.h5ad")
#Peak + GEX
print("Peak + GEX")
inter=ut.intersection([peak.obs.index, gex.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "GEX" : gex[inter]})
mudata.write_h5mu(f"Datasets/PBMC/CM/PBMC_Peak_GEX_Def.h5mu", compression="gzip")

Peak + GEX


In [15]:
mudata

MuData object with n_obs × n_vars = 5825 × 91513
  var:	'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
  2 modalities
    Peak:	5825 × 63906
      obs:	'RNA_celltype', 'Raw_CellType', 'CellType', 'nucleosome_signal', 'tss_enrichment_score', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'passes_filter'
      var:	'chr', 'start', 'stop', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'fragment_lengths', 'hvg', 'tss_enrichment'
      layers:	'counts'
    GEX:	5825 × 27607
      obs:	'RNA_celltype', 'Raw_CellType', 'CellType', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'mt_frac', 'ribofrac', 'passes_filter', 'doublet_score', 'predicted_doublet'
      var:	'gene_ids', 'feature_types', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'hvg', 'scrublet'
      layers:	'counts'

In [12]:
mudata = mu.read(f"Datasets/PBMC/CM/PBMC_Peak_GEX_Def.h5mu")

# MouseBrain

## GEX

In [31]:
adata=sc.read_h5ad("../GNN/Datasets/MouseBrain/FeatureSpaces/GEX/CM/MouseBrain_GEX_Raw.h5ad")
adata

AnnData object with n_obs × n_vars = 3365 × 32285
    obs: 'CellType'
    var: 'gene_ids', 'feature_types', 'genome', 'interval'

In [32]:
adata=ut.qc_filtering(adata, omic="GEX")

Adata's shape: (3365, 32285)
Adata's shape after cells and features filtering: (3028, 22367)
Adata's shape after MT filtering: (2744, 22367)
Adata's shape after doublets filtering: (2732, 22367)
Adata's shape after HVG filtering: (2732, 22367)


In [33]:
adata

AnnData object with n_obs × n_vars = 2732 × 22367
    obs: 'CellType', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'mt_frac', 'ribofrac', 'passes_filter', 'doublet_score', 'predicted_doublet'
    var: 'gene_ids', 'feature_types', 'genome', 'interval', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'scrublet', 'hvg'
    layers: 'counts'

In [35]:
adata.write("Datasets/MouseBrain/CM/MouseBrain_GEX_Def.h5ad", compression="gzip")

## Peak

In [38]:
adata=sc.read_h5ad("../GNN/Datasets/MouseBrain/FeatureSpaces/Peak/CM/MouseBrain_Peak_Raw.h5ad")
adata

AnnData object with n_obs × n_vars = 3365 × 160261
    obs: 'CellType', 'nucleosome_signal', 'tss_enrichment_score'
    var: 'chr', 'start', 'stop'
    uns: 'fragment_lengths', 'tss_enrichment'

In [39]:
adata=ut.qc_filtering(adata, omic="ATAC")

Adata's shape: (3365, 160261)
Adata's shape after cells and features filtering: (3027, 145569)
Adata's shape after HVG filtering: (3027, 145569)


In [40]:
adata

AnnData object with n_obs × n_vars = 3027 × 145569
    obs: 'CellType', 'nucleosome_signal', 'tss_enrichment_score', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'passes_filter'
    var: 'chr', 'start', 'stop', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'fragment_lengths', 'tss_enrichment', 'hvg'
    layers: 'counts'

In [41]:
adata.write("Datasets/MouseBrain/CM/MouseBrain_Peak_Def.h5ad", compression="gzip")

## Muon object

In [42]:
import muon as mu
peak=sc.read_h5ad(f"Datasets/MouseBrain/CM/MouseBrain_GEX_Def.h5ad")
gex=sc.read_h5ad(f"Datasets/MouseBrain/CM/MouseBrain_Peak_Def.h5ad")
#Peak + GEX
print("Peak + GEX")
inter=ut.intersection([peak.obs.index, gex.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "GEX" : gex[inter]})
mudata.write_h5mu(f"Datasets/MouseBrain/CM/MouseBrain_Peak_GEX_Def.h5mu", compression="gzip")

Peak + GEX


In [43]:
mudata

MuData object with n_obs × n_vars = 2500 × 167936
  var:	'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
  2 modalities
    Peak:	2500 x 22367
      obs:	'CellType', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'mt_frac', 'ribofrac', 'passes_filter', 'doublet_score', 'predicted_doublet'
      var:	'gene_ids', 'feature_types', 'genome', 'interval', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'hvg', 'scrublet'
      layers:	'counts'
    GEX:	2500 x 145569
      obs:	'CellType', 'nucleosome_signal', 'tss_enrichment_score', 'n_features', 'log_n_features', 'n_counts', 'log_n_counts', 'passes_filter'
      var:	'chr', 'start', 'stop', 'n_cells', 'log_n_cells', 'passes_filter', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
      uns:	'fragment_lengths', 'hvg', 'tss_enrichment'
      layers:	'counts'

# 2D

In [ ]:
for dataset in ["10XhsBrain3kMO","10XhsPBMC10kMO","MouseBrain"]:
    for fs in ["Peak","GEX"]:
        print(dataset, fs)https://github.com/KrishnaswamyLab/PHATE.git
        adata=sc.read_h5ad(f"Datasets/{dataset}/{fs}/{dataset}_{fs}_QC.h5ad")
        adata.var_names_make_unique()
        adata=adata[adata.obs["CellType"].dropna().index]
        mymap = dict([(y,str(x)) for x,y in enumerate(sorted(set(adata.obs["CellType"])))])
        inv_map = {v: k for k, v in mymap.items()}
        adata.uns["map"]=mymap
        adata.uns["inv_map"]=inv_map
        adata.obs["target"]=[mymap[x] for x in adata.obs["CellType"]]
        print(dataset, fs)
       
        phate_operator = phate.PHATE(n_jobs=-2, n_components=int(np.around(adata.shape[1]**(1/1.5), 0)))
        adata.obsm["PHATE_hidden"]=phate_operator.fit_transform(adata.X.toarray())
        
        phate_operator = phate.PHATE(n_jobs=-2, n_components=int(np.around(adata.shape[1]**(1/3), 0)))
        adata.obsm["PHATE_latent"]=phate_operator.fit_transform(adata.X.toarray())
    
        adata.write(f"Datasets/{dataset}/{fs}/{dataset}_{fs}_Def.h5ad", compression="gzip")

In [ ]:
for dataset in ["10XhsPBMC10kMO","MouseBrain"]:
    for fs in ["Peak","GEX"]:
        print(dataset, fs)
        adata=sc.read_h5ad(f"Datasets/{dataset}/{fs}/{dataset}_{fs}_Def.h5ad")
        mp_op = mp.Multiscale_PHATE(random_state=1, n_jobs=-2)
        levels = mp_op.fit(adata.X.toarray()) 
        for level in levels:
            adata.uns[f"MSPHATE_{level}"]=mp_op.transform(visualization_level = level, cluster_level = level)[0]

        phate_operator = phate.PHATE(n_jobs=-2, n_components=2)
        adata.obsm["PHATE"]=phate_operator.fit_transform(adata.X.toarray())
        
        adata.write(f"Datasets/{dataset}/{fs}/{dataset}_{fs}_MSPHATE.h5ad", compression="gzip")

In [ ]:
adata